# Паттерн 14: Map-Reduce — параллельный fan-out через Send()

`Send()` — механизм для создания динамического числа параллельных экземпляров узла. Каждый экземпляр — полноценный агент, который анализирует свою часть данных. Количество экземпляров определяется данными, а не структурой графа: три темы — три параллельных агента, десять тем — десять.

Результаты всех параллельных агентов собираются через reducer `operator.add` и передаются агрегатору, который синтезирует итоговый ответ.

```mermaid
graph LR
    START((START)) -->|split_topics| analyze1(["🔹 Analyze: тема 1<br/><i>обрабатывает часть данных</i>"])
    START -->|split_topics| analyze2(["🔹 Analyze: тема 2<br/><i>обрабатывает часть данных</i>"])
    START -->|split_topics| analyze3(["🔹 Analyze: тема 3<br/><i>обрабатывает часть данных</i>"])
    analyze1 --> aggregate(["📊 Aggregate<br/><i>объединяет результаты</i>"])
    analyze2 --> aggregate
    analyze3 --> aggregate
    aggregate --> END((END))

    classDef terminal fill:#95A5A6,stroke:#707B7C,color:#fff
    classDef worker fill:#2ECC71,stroke:#1A8B4C,color:#fff,stroke-width:2px
    classDef aggregator fill:#F59E0B,stroke:#D97706,color:#fff,stroke-width:2px

    class START,END terminal
    class analyze1,analyze2,analyze3 worker
    class aggregate aggregator
```

In [1]:
import sys
sys.path.insert(0, "..")

import operator
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langchain_core.messages import HumanMessage, SystemMessage
from llm_config import get_llm, check_api_key

In [2]:
if not check_api_key():
    raise ValueError("API key is not set")
else:
    print("API key is set")

API key is set


## Два класса состояний

Map-Reduce требует двух `TypedDict`: основное состояние графа и состояние для каждого параллельного экземпляра.

`MapReduceState` — состояние всего графа. `TopicState` — то, что получает каждый параллельный экземпляр. Поле `analyses` с reducer `operator.add` — точка сбора: каждый параллельный агент добавляет свой результат, и все они накапливаются в списке.

In [3]:
llm = get_llm()

# --- Основное состояние графа ---
class MapReduceState(TypedDict):
    question: str
    topics: list[str]
    analyses: Annotated[list[str], operator.add]
    final_answer: str

# --- Состояние для каждого параллельного экземпляра ---
class TopicState(TypedDict):
    question: str
    topic: str
    topic_index: int

## Функция fan-out

Функция-маппер возвращает список `Send()`, каждый из которых создаёт экземпляр узла с индивидуальным состоянием. Три темы — три параллельных экземпляра `analyze`. Десять тем — десять. Число определяется данными, а не структурой графа. Это и есть динамический fan-out.

In [4]:
def split_topics(state: MapReduceState) -> list[Send]:
    """
    Маппер: для каждой темы создаёт Send() к узлу analyze.
    Количество Send() = количество тем. Все запускаются параллельно.
    """
    topics = state["topics"]
    print(f"  [Маппер] Запускаю параллельный анализ {len(topics)} тем:")
    sends = []
    for i, topic in enumerate(topics):
        print(f"    → [{i + 1}] {topic}")
        sends.append(Send("analyze", {
            "question": state["question"],
            "topic": topic,
            "topic_index": i,
        }))
    return sends

## Узел-анализатор и агрегатор

Каждый экземпляр `analyze` получает `TopicState` с одной темой, вызывает LLM и возвращает результат в поле `analyses`. Reducer `operator.add` собирает все результаты в основном состоянии.

Агрегатор (fan-in узел) получает полный список `analyses` и синтезирует итоговый ответ. Он видит все результаты параллельных агентов и может выявить общие тренды, противоречия или ключевые выводы.

In [5]:
def analyze_node(state: TopicState) -> dict:
    """
    Каждый экземпляр анализирует одну тему.
    Результат добавляется в список analyses через reducer.
    """
    response = llm.invoke([
        SystemMessage(content=(
            "Ты — аналитик. Дай краткий, но содержательный анализ темы "
            "(3-4 предложения): ключевые тренды, текущее состояние, перспективы."
        )),
        HumanMessage(content=f"Общий вопрос: {state['question']}\n\nТвоя тема: {state['topic']}")
    ])

    result = f"**{state['topic']}**: {response.content}"
    print(f"  [Анализатор #{state['topic_index'] + 1}] {state['topic'][:40]} — готово")
    return {"analyses": [result]}


def aggregate_node(state: MapReduceState) -> dict:
    """
    Fan-in: собирает все анализы и синтезирует общий отчёт.
    Видит полный список результатов от всех параллельных агентов.
    """
    all_analyses = "\n\n".join(state["analyses"])

    response = llm.invoke([
        SystemMessage(content=(
            "Ты — главный аналитик. Синтезируй общий отчёт из нескольких "
            "тематических анализов. Выдели общие тренды, ключевые выводы "
            "и рекомендации. 4-5 предложений."
        )),
        HumanMessage(content=f"Вопрос: {state['question']}\n\nАнализы по темам:\n{all_analyses}")
    ])

    print(f"  [Агрегатор] Общий отчёт синтезирован")
    return {"final_answer": response.content}

## Сборка графа

`add_conditional_edges` с функцией, возвращающей `list[Send]`, — это точка fan-out. LangGraph автоматически создаёт нужное число экземпляров и ждёт завершения всех перед переходом к `aggregate`. После агрегации — безусловное ребро к `END`.

In [6]:
graph = StateGraph(MapReduceState)

graph.add_node("analyze", analyze_node)
graph.add_node("aggregate", aggregate_node)

# Fan-out: split_topics возвращает list[Send] → запуск параллельных analyze
graph.add_conditional_edges(START, split_topics)

# Fan-in: после всех analyze → агрегация
graph.add_edge("analyze", "aggregate")
graph.add_edge("aggregate", END)

app = graph.compile()

## Запуск

Передаём вопрос и список из четырёх тем. Маппер создаст четыре параллельных экземпляра `analyze` через `Send()`. Каждый вызовет LLM для анализа своей темы. После завершения всех четырёх агрегатор синтезирует общий отчёт.

In [7]:
result = app.invoke({
    "question": "Какие технологии определят развитие IT в ближайшие 5 лет?",
    "topics": [
        "Генеративный ИИ и мультиагентные системы",
        "Квантовые вычисления",
        "Робототехника и автономные системы",
        "Кибербезопасность в эпоху ИИ",
    ],
    "analyses": [],
    "final_answer": "",
})

print(f"\n  Проанализировано тем: {len(result['analyses'])}")
print(f"  Общий отчёт: {result['final_answer'][:200]}...")

  [Маппер] Запускаю параллельный анализ 4 тем:
    → [1] Генеративный ИИ и мультиагентные системы
    → [2] Квантовые вычисления
    → [3] Робототехника и автономные системы
    → [4] Кибербезопасность в эпоху ИИ


  [Анализатор #1] Генеративный ИИ и мультиагентные системы — готово
  [Анализатор #2] Квантовые вычисления — готово


  [Анализатор #4] Кибербезопасность в эпоху ИИ — готово
  [Анализатор #3] Робототехника и автономные системы — готово


  [Агрегатор] Общий отчёт синтезирован

  Проанализировано тем: 4
  Общий отчёт: В ближайшие 5 лет развитие IT будут определять прежде всего генеративный ИИ и мультиагентные системы, которые перейдут от роли помощников к автономному выполнению бизнес-задач и встроятся в операционн...


## Итог

Map-Reduce через `Send()` -- ключевые идеи:

- **Динамический fan-out**: количество параллельных экземпляров определяется данными на этапе выполнения, а не при сборке графа
- **Два класса состояний**: `MapReduceState` для всего графа и `TopicState` для каждого параллельного экземпляра
- **Reducer `operator.add`**: точка сбора результатов -- каждый агент добавляет свой результат в общий список
- **Автоматический fan-in**: LangGraph ждёт завершения всех параллельных экземпляров перед переходом к агрегатору